# sets
Sets are unordered collections of unique hashable values. In real systems they matter for membership checks, deduplication, blacklist logic, and fast comparison of categories or identities. Their real value is usually performance plus semantic clarity around uniqueness.

## 1. Problem First
Why does this matter in real systems?
- Deduplicating IDs is a common pipeline step.
- Membership checks against a blocklist or allowlist need to be fast.
- Set operations model overlap, difference, and missing coverage cleanly.

In [ ]:
user_ids = {101, 102, 103}
print(102 in user_ids)
print(999 in user_ids)

## 2. Minimal Concept
Core syntax:
- Create with `{1, 2, 3}` or `set(iterable)`
- Elements must be hashable
- Duplicates collapse automatically
- Common operations: union, intersection, difference, membership

## 3. Mental Model
How Python actually behaves
A set is a hash-table-backed structure optimized for uniqueness and fast membership. It does not preserve insertion order as a semantic guarantee you should depend on for logic. Its job is answering “is this here?” and “how do these groups overlap?”

In [ ]:
values = [1, 2, 2, 3, 3, 3]
unique_values = set(values)
print(unique_values)
print(len(unique_values))

## 4. Internal Mechanics
Set membership relies on hashing. That is why mutable unhashable objects like lists cannot be set elements. Identity is not the key feature here; hashability and equality are.

In [ ]:
import dis

def contains(blocked, item):
    return item in blocked

dis.dis(contains)
print(contains({"bad", "evil"}, "bad"))
try:
    { [1, 2] }
except TypeError as exc:
    print(type(exc).__name__)

## 5. Edge Cases
Where it breaks:
- Order should not be treated as meaningful application logic.
- Unhashable values like lists and dicts cannot be stored directly.
- Converting to a set destroys duplicates, which may lose business meaning.

In [ ]:
events = ["ERROR", "ERROR", "WARN"]
print(set(events))
print("deduplication removed frequency information")

## 6. Performance Thinking
- Average membership check is O(1).
- Union, intersection, and difference are usually linear in input sizes.
- A set is often the right replacement for repeated `x in list` lookups.
- Memory usage is higher than a bare list because hashing infrastructure exists.

## 7. Real Use Case
A security filter may use a set of blocked IPs or tokens for fast rejection during request handling.

In [ ]:
blocked_ips = {"10.0.0.1", "10.0.0.9"}
incoming_ip = "10.0.0.9"
if incoming_ip in blocked_ips:
    print("reject")
else:
    print("allow")

## 8. Anti-Pattern
What beginners do wrong:
- Convert lists to sets without checking whether duplicates carry meaning.
- Depend on output order from sets.
- Use lists for heavy membership checks instead of sets.

In [ ]:
allowed = ["read", "write", "deploy"]
print("deploy" in allowed)
allowed_set = set(allowed)
print("deploy" in allowed_set)

## 9. Interview Signals
Questions FAANG asks:
- Why are set lookups usually faster than list lookups?
- What kinds of values can go in a set?
- What information is lost when converting a list to a set?
- When should you use set operations instead of manual loops?

## 10. Exercise (Non-trivial)
Design a deduplication and access-control step for incoming events. Keep a set of seen IDs to detect duplicates, a set of blocked actors for fast rejection, and explain what additional structure you would need if frequency still mattered.

In [ ]:
def filter_events(events, blocked_ids):
    seen = set()
    result = []
    for event in events:
        if event["id"] in blocked_ids or event["id"] in seen:
            continue
        seen.add(event["id"])
        result.append(event)
    return result

# Task:
# 1. Explain the role of each set.
# 2. Identify the time complexity of membership checks.
# 3. Describe how to preserve duplicate counts if needed.